# Load Libraries

In [19]:
import os
import warnings
import logging
import sys
import pickle
import numpy as np
import pandas as pd
import geopandas as gpd
import dotenv
import pyet
import matplotlib.pyplot as plt


# Load environment variables from .env file
dotenv.load_dotenv()

# Set up logging
logging.basicConfig(level=logging.INFO)

# Suppress warnings
warnings.filterwarnings("ignore")

# Set display options for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', None)

# Load Data

In [1]:
# # Load Data From Pickle
# with open('../../data/Iran_Daily_Data_1951_2025.pkl', 'rb') as f:
#     data = pickle.load(f)
#     logging.info("Data loaded from pickle file.")

# Load Parquet Data
data = pd.read_parquet('../../data/Iran_Daily_Data_1951_2025.parquet')
logging.info("Data loaded from parquet file.")    

NameError: name 'pd' is not defined

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21097489 entries, 0 to 21097488
Data columns (total 26 columns):
 #   Column             Dtype         
---  ------             -----         
 0   date               datetime64[ns]
 1   station_id         object        
 2   station_name       object        
 3   region_id          object        
 4   region_name        object        
 5   lat                float64       
 6   lon                float64       
 7   station_elevation  float64       
 8   tmax               float64       
 9   tmin               float64       
 10  tm                 float64       
 11  umax               float64       
 12  umin               float64       
 13  um                 float64       
 14  ffm                float64       
 15  sshn               float64       
 16  rrr24              float64       
 17  pm                 float64       
 18  p0m                float64       
 19  ewm                float64       
 20  radglo24           flo

# Calculate ETo

#### Data Quality Control

In [22]:
def data_quality_control(
    df, 
    required_vars = ['tm']
):
    df.dropna(subset=required_vars, inplace=True, how='any')
    return df

#### Calculate ETo

In [23]:
# Potential Evapotranspiration Estimated Based On All Available Methods
def calculate_all_eto(group):
    print(f"Calculating All ETo Methods For Station {group.name}")
    # group = data_quality_control(
    #     df=group, 
    #     required_vars=['tm', 'tmax', 'tmin', 'um', 'umin', 'umax', 'ffm', 'sshn', 'lat', 'station_elevation']
    # )
    group = group.set_index('date').sort_index()

    if group.empty:
        return pd.Series(np.nan, index=group.index)

    lat = pyet.utils.deg_to_rad(lat=group['lat'].iloc[0])
    elevation = group['station_elevation'].iloc[0]
    rs = pyet.calc_rad_sol_in(n=group['sshn'], lat=lat, as1=0.25, bs1=0.5, nn=None)    
    wind = group['ffm']
    tmean = group['tm']
    tmax = group["tmax"]
    tmin = group["tmin"]
    rh = group["um"]
    rhmax = group["umin"]
    rhmin = group["umax"]
    
    eto = pyet.calculate_all(
        tmean=tmean,
        wind=wind,
        rs=rs,
        tmax=tmax,
        tmin=tmin,
        rhmax=rhmax,
        rhmin=rhmin,
        rh=rh,
        elevation=elevation,
        lat=lat,
    )

    return eto




# # ASCE-PM: pm_asce
# def calculate_eto_pm_asce(group):
#     print(f"Calculating ETo pm_asce for station {group.name}")
#     group = data_quality_control(
#         df=group, 
#         required_vars=['tm', 'tmax', 'tmin', 'um', 'umin', 'umax', 'ffm', 'sshn', 'lat', 'station_elevation']
#     )
#     group = group.set_index('date').sort_index()

#     # If group is empty after cleaning, return NaNs
#     if group.empty:
#         return pd.Series(np.nan, index=group.index)

#     lat = pyet.utils.deg_to_rad(lat=group['lat'].iloc[0])
#     elevation = group['station_elevation'].iloc[0]
#     rs = pyet.calc_rad_sol_in(n=group['sshn'], lat=lat, as1=0.25, bs1=0.5, nn=None)    
#     wind = group['ffm']
#     tmean = group['tm']
#     tmax = group["tmax"]
#     tmin = group["tmin"]
#     rh = group["um"]
#     rhmax = group["umin"]
#     rhmin = group["umax"]
    
#     eto = pyet.pm_asce(
#         tmean=tmean,
#         wind=wind,
#         rs=rs,
#         rn=None,
#         g=0,
#         tmax=tmax,
#         tmin=tmin,
#         rhmax=rhmax,
#         rhmin=rhmin,
#         rh=rh,
#         pressure=None,
#         elevation=elevation,
#         lat=lat,
#         n=None,
#         nn=None,
#         rso=None,
#         a=1.35,
#         b=-0.35,
#         cn=900,
#         cd=0.34,
#         ea=None,
#         albedo=0.23,
#         kab=None,
#         as1=0.25,
#         bs1=0.5,
#         clip_zero=True,
#         etype="os",
#     )

#     return eto


# # FAO-56: pm_fao56
# def calculate_eto_pm_fao56(group):
#     print(f"Calculating ETo pm_fao56 for station {group.name}")
#     group = data_quality_control(
#         df=group, 
#         required_vars=['tm', 'tmax', 'tmin', 'um', 'umin', 'umax', 'ffm', 'sshn', 'lat', 'station_elevation']
#     )
#     group = group.set_index('date').sort_index()

#     # If group is empty after cleaning, return NaNs
#     if group.empty:
#         return pd.Series(np.nan, index=group.index)

#     lat = pyet.utils.deg_to_rad(lat=group['lat'].iloc[0])
#     elevation = group['station_elevation'].iloc[0]
#     rs = pyet.calc_rad_sol_in(n=group['sshn'], lat=lat, as1=0.25, bs1=0.5, nn=None)    
#     wind = group['ffm']
#     tmean = group['tm']
#     tmax = group["tmax"]
#     tmin = group["tmin"]
#     rh = group["um"]
#     rhmax = group["umin"]
#     rhmin = group["umax"]
    
#     eto = pyet.pm_fao56(
#         tmean=tmean,
#         wind=wind,
#         rs=rs,
#         elevation=elevation,
#         lat=lat,
#         tmax=tmax,
#         tmin=tmin,
#         rh=rh,
#         rhmax=rhmax,
#         rhmin=rhmin
#     )

#     return eto

# # Hargreaves: hargreaves
# def calculate_eto_hargreaves(group):
#     print(f"Calculating ETo hargreaves for station {group.name}")
#     group = data_quality_control(
#         df=group, 
#         required_vars=['tm', 'tmax', 'tmin', 'lat']
#     )
#     group = group.set_index('date').sort_index()

#     # If group is empty after cleaning, return NaNs
#     if group.empty:
#         return pd.Series(np.nan, index=group.index)

#     lat = pyet.utils.deg_to_rad(lat=group['lat'].iloc[0])
#     tmean = group['tm']
#     tmax = group["tmax"]
#     tmin = group["tmin"]
    
#     eto = pyet.hargreaves(
#         tmean=tmean,
#         tmax=tmax,
#         tmin=tmin,
#         lat=lat,
#         k=0.0135,
#         method=0,
#         clip_zero=True
#     )

#     return eto

# # Blaney-Criddle: blaney_criddle
# def calculate_eto_blaney_criddle(group):
#     print(f"Calculating ETo blaney_criddle for station {group.name}")
#     group = data_quality_control(
#         df=group, 
#         required_vars=['tm', 'lat']
#     )
#     group = group.set_index('date').sort_index()

#     # If group is empty after cleaning, return NaNs
#     if group.empty:
#         return pd.Series(np.nan, index=group.index)

#     lat = pyet.utils.deg_to_rad(lat=group['lat'].iloc[0])
#     tmean = group['tm']
    
#     eto = pyet.blaney_criddle(
#         tmean=tmean,
#         lat=lat,
#     )

#     return eto

# # Oudin: oudin
# def calculate_eto_oudin(group):
#     print(f"Calculating ETo oudins for station {group.name}")
#     group = data_quality_control(
#         df=group, 
#         required_vars=['tm', 'lat']
#     )
#     group = group.set_index('date').sort_index()

#     # If group is empty after cleaning, return NaNs
#     if group.empty:
#         return pd.Series(np.nan, index=group.index)

#     tmean = group['tm']
#     lat = pyet.utils.deg_to_rad(lat=group['lat'].iloc[0])
    
#     eto = pyet.oudin(
#         tmean=tmean,
#         lat=lat,
#         k1=100, 
#         k2=5, 
#         clip_zero=True
#     )

#     return eto

# # Makkink: makkink
# def calculate_eto_makkink(group):
#     print(f"Calculating ETo makkink for station {group.name}")
#     group = data_quality_control(
#         df=group, 
#         required_vars=['tm', 'sshn', 'station_elevation']
#     )
#     group = group.set_index('date').sort_index()

#     # If group is empty after cleaning, return NaNs
#     if group.empty:
#         return pd.Series(np.nan, index=group.index)

#     tmean = group['tm']
#     lat = pyet.utils.deg_to_rad(lat=group['lat'].iloc[0])
#     rs = pyet.calc_rad_sol_in(n=group['sshn'], lat=lat, as1=0.25, bs1=0.5, nn=None) 
#     elevation = group['station_elevation'].iloc[0]
    
#     eto = pyet.makkink(
#         tmean=tmean,
#         rs=rs,
#         pressure=None,
#         elevation=elevation,
#         k=0.65,
#         clip_zero=True
#     )

#     return eto


# eto_df_pm_asce = data.groupby(
#     ['region_id', 'region_name', 'station_id', 'station_name'],
#     group_keys=True
# ).apply(calculate_eto_pm_asce)

# eto_df_pm_fao56 = data.groupby(
#     ['region_id', 'region_name', 'station_id', 'station_name'],
#     group_keys=True
# ).apply(calculate_eto_pm_fao56)

# eto_df_hargreaves = data.groupby(
#     ['region_id', 'region_name', 'station_id', 'station_name'],
#     group_keys=True
# ).apply(calculate_eto_hargreaves)

# eto_df_blaney_criddle = data.groupby(
#     ['region_id', 'region_name', 'station_id', 'station_name'],
#     group_keys=True
# ).apply(calculate_eto_blaney_criddle)

# eto_df_oudin = data.groupby(
#     ['region_id', 'region_name', 'station_id', 'station_name'],
#     group_keys=True
# ).apply(calculate_eto_oudin)

# eto_df_makkink = data.groupby(
#     ['region_id', 'region_name', 'station_id', 'station_name'],
#     group_keys=True
# ).apply(calculate_eto_makkink)

all_eto_df = data.groupby(
    ['region_id', 'region_name', 'station_id', 'station_name'],
    group_keys=True
).apply(calculate_all_eto)

Calculating All ETo Methods For Station ('ALKK', 'Alborz', '18554', 'Zidashte Taleghan')


Exception: The maximum value of relative humidity provided is smaller than 1 [%], which is not realistic. Please convert the relative humidity to [%].

In [7]:
all_eto_df.reset_index()

,region_id,region_name,station_id,station_name,date,0,Penman,PM,PM-ASCE,FAO-56,Priestley-Taylor,Kimberly-Penman,Thom-Oliver,Blaney-Criddle,Hamon,Romanenko,Linacre,Haude,Turc,Jensen-Haise,Mcguinness-Bordne,Hargreaves,FAO-24,Abtew,Makkink,Oudin
0,ALKK,Alborz,40752,Karaj,1985-01-01 12:00:00,NaN,0.48,0.44,0.45,0.45,0.51,0.42,0.54,0.00,0.35,0.63,1.02,0.39,0.00,0.00,0.13,0.78,0.18,1.12,0.52,0.09
1,ALKK,Alborz,40752,Karaj,1985-01-03 12:00:00,NaN,0.51,0.48,0.49,0.49,0.57,0.47,0.49,0.00,0.34,0.38,0.57,0.28,0.00,0.00,0.06,0.65,0.46,1.89,0.85,0.04
2,ALKK,Alborz,40752,Karaj,1985-01-04 12:00:00,NaN,0.65,0.70,0.72,0.72,0.62,0.54,0.73,0.00,0.39,1.29,1.56,0.73,0.00,0.07,0.26,0.95,0.86,2.22,1.09,0.18
3,ALKK,Alborz,40752,Karaj,1985-01-05 12:00:00,NaN,0.55,0.45,0.46,0.46,0.57,0.45,0.70,0.00,0.36,1.08,0.91,0.69,0.00,0.00,0.15,0.74,0.60,1.90,0.88,0.10
4,ALKK,Alborz,40752,Karaj,1985-01-06 12:00:00,NaN,0.58,0.54,0.55,0.55,0.64,0.52,0.57,0.00,0.34,0.52,1.14,0.43,0.00,0.00,0.07,0.79,0.76,2.43,1.10,0.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3112923,QOQM,Qom,99440,Kahak,2025-09-17 12:00:00,NaN,4.66,6.54,6.52,6.52,4.11,4.30,5.55,3.52,3.14,11.42,6.41,10.52,6.13,6.04,5.52,4.73,5.61,4.73,4.40,3.75
3112924,QOQM,Qom,99440,Kahak,2025-09-18 12:00:00,NaN,4.62,6.76,6.74,6.74,3.90,4.25,5.65,3.47,3.07,12.26,6.28,10.49,6.23,5.98,5.42,4.62,5.64,4.73,4.39,3.69
3112925,QOQM,Qom,99440,Kahak,2025-09-19 12:00:00,NaN,4.75,7.48,7.43,7.43,3.76,4.31,6.26,3.79,3.63,14.78,7.68,14.91,6.61,6.28,5.92,5.56,5.62,4.50,4.31,4.03
3112926,QOQM,Qom,99440,Kahak,2025-09-20 12:00:00,NaN,4.64,7.27,7.23,7.23,3.68,4.23,6.07,3.71,3.47,14.97,7.06,12.50,6.87,6.50,5.77,4.63,5.91,4.75,4.53,3.92


#### Join ETo with Original Data

In [10]:
if isinstance(all_eto_df.index, pd.MultiIndex):
    all_eto_df = all_eto_df.reset_index().drop(columns=[0])
    data = pd.merge(data, all_eto_df, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
else:
    data = data.join(all_eto_df)

data

,date,station_id,station_name,region_id,region_name,lat,lon,station_elevation,tmax,tmin,tm,umax,umin,um,ffm,sshn,rrr24,pm,p0m,ewm,radglo24,evt,td_m,twet_m,tsoil_m,ewsm,Penman,PM,PM-ASCE,FAO-56,Priestley-Taylor,Kimberly-Penman,Thom-Oliver,Blaney-Criddle,Hamon,Romanenko,Linacre,Haude,Turc,Jensen-Haise,Mcguinness-Bordne,Hargreaves,FAO-24,Abtew,Makkink,Oudin
0,1951-01-01 12:00:00,18554,Zidashte Taleghan,ALKK,Alborz,36.13,50.68,2255.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1951-01-02 12:00:00,18554,Zidashte Taleghan,ALKK,Alborz,36.13,50.68,2255.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1951-01-03 12:00:00,18554,Zidashte Taleghan,ALKK,Alborz,36.13,50.68,2255.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1951-01-04 12:00:00,18554,Zidashte Taleghan,ALKK,Alborz,36.13,50.68,2255.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1951-01-05 12:00:00,18554,Zidashte Taleghan,ALKK,Alborz,36.13,50.68,2255.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21097484,2025-09-17 12:00:00,99440,Kahak,QOQM,Qom,34.40,50.87,1403.10,31.40,16.90,24.10,41.00,18.00,26.17,3.17,10.70,0.00,1005.84,858.92,7.92,2348.00,10.40,3.42,13.44,16.00,31.74,4.66,6.54,6.52,6.52,4.11,4.30,5.55,3.52,3.14,11.42,6.41,10.52,6.13,6.04,5.52,4.73,5.61,4.73,4.40,3.75
21097485,2025-09-18 12:00:00,99440,Kahak,QOQM,Qom,34.40,50.87,1403.10,30.90,16.70,23.80,33.00,18.00,24.25,3.25,10.80,0.00,1005.85,858.89,7.56,2351.00,9.90,2.72,13.30,15.00,31.94,4.62,6.76,6.74,6.74,3.90,4.25,5.65,3.47,3.07,12.26,6.28,10.49,6.23,5.98,5.42,4.62,5.64,4.73,4.39,3.69
21097486,2025-09-19 12:00:00,99440,Kahak,QOQM,Qom,34.40,50.87,1403.10,35.70,17.50,26.60,28.00,8.00,17.70,3.13,10.00,0.00,1000.59,855.40,6.28,2247.00,10.40,0.25,13.68,16.00,38.07,4.75,7.48,7.43,7.43,3.76,4.31,6.26,3.79,3.63,14.78,7.68,14.91,6.61,6.28,5.92,5.56,5.62,4.50,4.31,4.03
21097487,2025-09-20 12:00:00,99440,Kahak,QOQM,Qom,34.40,50.87,1403.10,32.60,19.40,26.00,27.00,5.00,18.00,3.14,11.00,0.00,1000.94,855.31,5.15,2341.00,10.60,-3.18,11.51,16.00,31.17,4.64,7.27,7.23,7.23,3.68,4.23,6.07,3.71,3.47,14.97,7.06,12.50,6.87,6.50,5.77,4.63,5.91,4.75,4.53,3.92


In [ ]:
# if isinstance(eto_df_pm_asce.index, pd.MultiIndex):
#     eto_df_pm_asce = eto_df_pm_asce.reset_index().rename(columns={0:'ASCE_PM'})
#     data = pd.merge(data, eto_df_pm_asce, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
# else:
#     data = data.join(eto_df_pm_asce)
    
# if isinstance(eto_df_pm_fao56.index, pd.MultiIndex):
#     eto_df_pm_fao56 = eto_df_pm_fao56.reset_index().rename(columns={0:'FAO56'})
#     data = pd.merge(data, eto_df_pm_fao56, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
# else:
#     data = data.join(eto_df_pm_fao56)

# if isinstance(eto_df_hargreaves.index, pd.MultiIndex):
#     eto_df_hargreaves = eto_df_hargreaves.reset_index().rename(columns={0:'Hargreaves'})
#     data = pd.merge(data, eto_df_hargreaves, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
# else:
#     data = data.join(eto_df_hargreaves)

# if isinstance(eto_df_blaney_criddle.index, pd.MultiIndex):
#     eto_df_blaney_criddle = eto_df_blaney_criddle.reset_index().rename(columns={0:'Blaney_Criddle'})
#     data = pd.merge(data, eto_df_blaney_criddle, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
# else:
#     data = data.join(eto_df_blaney_criddle)

# if isinstance(eto_df_oudin.index, pd.MultiIndex):
#     eto_df_oudin = eto_df_oudin.reset_index().rename(columns={0:'Oudin'})
#     data = pd.merge(data, eto_df_oudin, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
# else:
#     data = data.join(eto_df_oudin)

# if isinstance(eto_df_thornthwaite.index, pd.MultiIndex):
#     eto_df_thornthwaite = eto_df_thornthwaite.reset_index().rename(columns={0:'Thornthwaite'})
#     data = pd.merge(data, eto_df_thornthwaite, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
# else:
#     data = data.join(eto_df_thornthwaite)

# if isinstance(eto_df_makkink.index, pd.MultiIndex):
#     eto_df_makkink = eto_df_makkink.reset_index().rename(columns={0:'Makkink'})
#     data = pd.merge(data, eto_df_makkink, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
# else:
#     data = data.join(eto_df_makkink)

# data

,date,station_id,station_name,region_id,region_name,lat,lon,station_elevation,tmax,tmin,tm,umax,umin,um,ffm,sshn,rrr24,pm,p0m,ewm,radglo24,evt,td_m,twet_m,tsoil_m,ewsm
0,1951-01-01 12:00:00,18554,Zidashte Taleghan,ALKK,Alborz,36.13,50.68,2255.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1951-01-02 12:00:00,18554,Zidashte Taleghan,ALKK,Alborz,36.13,50.68,2255.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1951-01-03 12:00:00,18554,Zidashte Taleghan,ALKK,Alborz,36.13,50.68,2255.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1951-01-04 12:00:00,18554,Zidashte Taleghan,ALKK,Alborz,36.13,50.68,2255.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1951-01-05 12:00:00,18554,Zidashte Taleghan,ALKK,Alborz,36.13,50.68,2255.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21097484,2025-09-17 12:00:00,99440,Kahak,QOQM,Qom,34.40,50.87,1403.10,31.40,16.90,24.10,41.00,18.00,26.17,3.17,10.70,0.00,1005.84,858.92,7.92,2348.00,10.40,3.42,13.44,16.00,31.74
21097485,2025-09-18 12:00:00,99440,Kahak,QOQM,Qom,34.40,50.87,1403.10,30.90,16.70,23.80,33.00,18.00,24.25,3.25,10.80,0.00,1005.85,858.89,7.56,2351.00,9.90,2.72,13.30,15.00,31.94
21097486,2025-09-19 12:00:00,99440,Kahak,QOQM,Qom,34.40,50.87,1403.10,35.70,17.50,26.60,28.00,8.00,17.70,3.13,10.00,0.00,1000.59,855.40,6.28,2247.00,10.40,0.25,13.68,16.00,38.07
21097487,2025-09-20 12:00:00,99440,Kahak,QOQM,Qom,34.40,50.87,1403.10,32.60,19.40,26.00,27.00,5.00,18.00,3.14,11.00,0.00,1000.94,855.31,5.15,2341.00,10.60,-3.18,11.51,16.00,31.17


In [ ]:
import matplotlib.pyplot as plt

stations_to_plot = data['station_id'].drop_duplicates().sample(1)

plt.figure(figsize=(12, 6))
for sid in stations_to_plot:
    station_data = data[data['station_id'] == sid].sort_values('date')
    # plt.plot(station_data['date'], station_data['FAO56'], label=f'Station {sid} - FAO56')
    plt.scatter(station_data['date'], station_data['Hargreaves'], color='red', label=f'Station {sid} - Hargreaves')
    plt.scatter(station_data['date'], station_data['FAO56'], color='blue', label=f'Station {sid} - FAO56', s=5)
    plt.scatter(station_data['date'], station_data['Blaney_Criddle'], color='green', label=f'Station {sid} - Blaney_Criddle', s=5)
    plt.scatter(station_data['date'], station_data['Oudin'], color='purple', label=f'Station {sid} - Oudin', s=5)

plt.xlabel('Date')
plt.ylabel('ETo')
plt.title('ETo Time Series for Selected Stations')
plt.legend()
plt.tight_layout()
plt.show()


# Change Column Names

In [ ]:
# result.rename(
#     columns={
#         'FAO-56': 'FAO56',
#         'Priestley-Taylor': 'PriestleyTaylor',
#         'Kimberly-Penman': 'KimberlyPenman',
#         'Thom-Oliver': 'ThomOliver',
#         'Blaney-Criddle': 'BlaneyCriddle',
#         'Jensen-Haise': 'JensenHaise',
#         'Mcguinness-Bordne': 'McguinnessBordne',
#         'FAO-24': 'FAO24',
#     }, 
#     inplace=True
# )

# Export Data

In [12]:
# # To pickle in data folder
# with open('../../data/Iran_Daily_ETo_1951_2025.pkl', 'wb') as f:
#     pickle.dump(data, f)

# To parquet in data folder
data.to_parquet('../../data/Iran_Daily_ETo_1951_2025.parquet')
logging.info("ETo data saved to pickle and parquet files.")

INFO:root:ETo data saved to pickle and parquet files.


# Monthly ETo Calculations (Daily Mean)

In [13]:
data.columns

Index(['date', 'station_id', 'station_name', 'region_id', 'region_name', 'lat',
       'lon', 'station_elevation', 'tmax', 'tmin', 'tm', 'umax', 'umin', 'um',
       'ffm', 'sshn', 'rrr24', 'pm', 'p0m', 'ewm', 'radglo24', 'evt', 'td_m',
       'twet_m', 'tsoil_m', 'ewsm', 'Penman', 'PM', 'PM-ASCE', 'FAO-56',
       'Priestley-Taylor', 'Kimberly-Penman', 'Thom-Oliver', 'Blaney-Criddle',
       'Hamon', 'Romanenko', 'Linacre', 'Haude', 'Turc', 'Jensen-Haise',
       'Mcguinness-Bordne', 'Hargreaves', 'FAO-24', 'Abtew', 'Makkink',
       'Oudin'],
      dtype='object')

In [14]:
data['year'] = data['date'].dt.year
data['month'] = data['date'].dt.month

monthly_data = data.groupby(
    ['region_id', 'region_name', 'station_id', 'station_name', 'year', 'month'],
    group_keys=True
).agg({
    'date': 'count',
    'lat': 'first',
    'lon': 'first',
    'station_elevation': 'first',
    'tmax': ['mean', 'count'],
    'tmin': ['mean', 'count'],
    'tm': ['mean', 'count'],
    'umax': ['mean', 'count'],
    'umin': ['mean', 'count'],
    'um': ['mean', 'count'],
    'ffm': ['mean', 'count'],
    'sshn': ['mean', 'count'],
    'rrr24': ['sum', 'count'],
    'pm': ['mean', 'count'],
    'p0m': ['mean', 'count'], 
    'ewm': ['mean', 'count'], 
    'radglo24': ['sum', 'count'],
    'evt': ['sum', 'count'],
    'td_m': ['mean', 'count'],
    'twet_m': ['mean', 'count'],
    'tsoil_m': ['mean', 'count'],
    'ewsm': ['mean', 'count'], 
    'Penman': ['sum', 'count'], 
    'PM': ['sum', 'count'], 
    'PM-ASCE': ['sum', 'count'],
    'FAO-56': ['sum', 'count'],
    'Priestley-Taylor': ['sum', 'count'],
    'Kimberly-Penman': ['sum', 'count'],
    'Thom-Oliver': ['sum', 'count'],
    'Blaney-Criddle': ['sum', 'count'],
    'Hamon': ['sum', 'count'], 
    'Romanenko': ['sum', 'count'],
    'Linacre': ['sum', 'count'],
    'Haude': ['sum', 'count'],
    'Turc': ['sum', 'count'],
    'Jensen-Haise': ['sum', 'count'],
    'Mcguinness-Bordne': ['sum', 'count'],
    'Hargreaves': ['sum', 'count'],
    'FAO-24': ['sum', 'count'],
    'Abtew': ['sum', 'count'],
    'Makkink': ['sum', 'count'],
    'Oudin': ['sum', 'count'],
})
monthly_data.columns = ['_'.join(col).strip() for col in monthly_data.columns.values]
monthly_data.columns = [col.replace('_first', '').replace('_mean', '').replace('_sum', '') for col in monthly_data.columns]
monthly_data = monthly_data.reset_index()
# monthly_data = monthly_data[[
#     'region_id', 'region_name', 'station_id', 'station_name', 'lon', 'lat', 'station_elevation',
#     'year', 'month',
#     'tm', 'tm_count',
#     'tmax', 'tmax_count',
#     'tmin', 'tmin_count',
#     'um', 'um_count',
#     'umax', 'umax_count',
#     'umin', 'umin_count',
#     'ffm', 'ffm_count',
#     'sshn', 'sshn_count',
#     'rrr24', 'rrr24_count',
#     'ASCE_PM', 'ASCE_PM_count',
#     'FAO56', 'FAO56_count',
#     'Hargreaves', 'Hargreaves_count',
#     'Blaney_Criddle', 'Blaney_Criddle_count',
#     'Oudin', 'Oudin_count',
#     'Thornthwaite', 'Thornthwaite_count',
#     'Makkink', 'Makkink_count',
#     'date_count'
# ]]

monthly_data = monthly_data.round(2)

# if *_count = 0, set the corresponding mean/sum to NaN
for col in monthly_data.columns:
    if col.endswith('_count'):
        if col == 'date_count':
            continue
        mean_col = col.replace('_count', '')
        monthly_data.loc[monthly_data[col] == 0, mean_col] = np.nan

# percent of day in month with data
for col in monthly_data.columns:
    if col.endswith('_count'):
        if col == 'date_count':
            continue
        monthly_data[col] = (monthly_data[col] / monthly_data['date_count']) * 100
        monthly_data[col] = monthly_data[col].round(2)

monthly_data.columns = [col.replace('_count', '_percent') for col in monthly_data.columns]

monthly_data.drop(columns=['date_percent'], inplace=True)

monthly_data

,region_id,region_name,station_id,station_name,year,month,lat,lon,station_elevation,tmax,tmax_percent,tmin,tmin_percent,tm,tm_percent,umax,umax_percent,umin,umin_percent,um,um_percent,ffm,ffm_percent,sshn,sshn_percent,rrr24,rrr24_percent,pm,pm_percent,p0m,p0m_percent,ewm,ewm_percent,radglo24,radglo24_percent,evt,evt_percent,td_m,td_m_percent,twet_m,twet_m_percent,tsoil_m,tsoil_m_percent,ewsm,ewsm_percent,Penman,Penman_percent,PM,PM_percent,PM-ASCE,PM-ASCE_percent,FAO-56,FAO-56_percent,Priestley-Taylor,Priestley-Taylor_percent,Kimberly-Penman,Kimberly-Penman_percent,Thom-Oliver,Thom-Oliver_percent,Blaney-Criddle,Blaney-Criddle_percent,Hamon,Hamon_percent,Romanenko,Romanenko_percent,Linacre,Linacre_percent,Haude,Haude_percent,Turc,Turc_percent,Jensen-Haise,Jensen-Haise_percent,Mcguinness-Bordne,Mcguinness-Bordne_percent,Hargreaves,Hargreaves_percent,FAO-24,FAO-24_percent,Abtew,Abtew_percent,Makkink,Makkink_percent,Oudin,Oudin_percent
0,ALKK,Alborz,18554,Zidashte Taleghan,1951,1,36.13,50.68,2255.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00
1,ALKK,Alborz,18554,Zidashte Taleghan,1951,2,36.13,50.68,2255.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00
2,ALKK,Alborz,18554,Zidashte Taleghan,1951,3,36.13,50.68,2255.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00
3,ALKK,Alborz,18554,Zidashte Taleghan,1951,4,36.13,50.68,2255.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00
4,ALKK,Alborz,18554,Zidashte Taleghan,1951,5,36.13,50.68,2255.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
693376,QOQM,Qom,99440,Kahak,2025,5,34.40,50.87,1403.10,32.10,100.00,18.04,100.00,25.07,100.00,36.13,100.00,14.87,100.00,23.45,100.00,3.39,100.00,10.69,96.77,4.21,100.00,1004.42,100.00,858.08,100.00,7.22,100.00,81513.00,96.77,356.40,100.00,1.72,100.00,13.41,100.00,15.55,100.00,34.00,100.00,175.30,96.77,236.28,96.77,235.12,96.77,235.12,96.77,161.65,96.77,154.27,96.77,200.66,96.77,130.11,96.77,130.13,96.77,379.65,96.77,202.06,96.77,429.83,96.77,220.88,96.77,217.93,96.77,215.80,96.77,180.25,96.77,202.95,96.77,165.82,96.77,155.67,96.77,146.81,96.77
693377,QOQM,Qom,99440,Kahak,2025,6,34.40,50.87,1403.10,33.87,100.00,20.21,100.00,27.04,100.00,33.67,100.00,15.43,100.00,22.38

In [15]:
# # To pickle in data folder
# with open('../../data/Iran_Monthly_Daily_Mean_ETo_1951_2025.pkl', 'wb') as f:
#     pickle.dump(monthly_data, f)

# To parquet in data folder
monthly_data.to_parquet('../../data/Iran_Monthly_Daily_Sum_ETo_1951_2025.parquet')
logging.info("Monthly ETo data saved to pickle and parquet files.")

INFO:root:Monthly ETo data saved to pickle and parquet files.


# Monthly ETo Calculations

In [16]:
# # Load Data From Pickle
# with open('../../data/Iran_Monthly_Data_1951_2025.pkl', 'rb') as f:
#     data = pickle.load(f)
#     logging.info("Data loaded from pickle file.")

# Load Parquet Data
data = pd.read_parquet('../../data/Iran_Monthly_Data_1951_2025.parquet')
logging.info("Data loaded from parquet file.")    

INFO:root:Data loaded from parquet file.


In [18]:
data

,year,month,region_id,region_name,station_id,station_name,lat,lon,station_elevation,tmax,tmax_count,tmin,tmin_count,tm,tm_count,umax,umax_count,umin,umin_count,um,um_count,ffm,ffm_count,sshn,sshn_count,pm,pm_count,p0m,p0m_count,ewm,ewm_count,radglo24,radglo24_count,td_m,td_m_count,twet_m,twet_m_count,tsoil_m,tsoil_m_count,ewsm,ewsm_count,evt,evt_count,rrr24,rrr24_count,date
0,1951,1,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,0.00,0,1951-01-15
1,1951,2,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,0.00,0,1951-02-15
2,1951,3,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,0.00,0,1951-03-15
3,1951,4,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,0.00,0,1951-04-15
4,1951,5,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,0.00,0,1951-05-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
693376,2025,5,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,24.46,13,12.53,10,19.07,10,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,12.20,11,2025-05-15
693377,2025,6,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,32.03,30,12.43,28,22.12,28,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,0.00,29,2025-06-15
693378,2025,7,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,36.54,31,17.80,30,27.14,30,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,3.10,30,2025-07-15
693379,2025,8,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,37.20,20,19.91,11,29.04,11,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,0.00,19,2025-08-15


In [17]:
# create a datetime column from year and month
data['date'] = pd.to_datetime(data[['year', 'month']].assign(day=15))

# Calculate Monthly Mean ETo

In [ ]:
eto_df_pm_asce = data.groupby(
    ['region_id', 'region_name', 'station_id', 'station_name'],
    group_keys=True
).apply(calculate_eto_pm_asce)

eto_df_pm_fao56 = data.groupby(
    ['region_id', 'region_name', 'station_id', 'station_name'],
    group_keys=True
).apply(calculate_eto_pm_fao56)

eto_df_hargreaves = data.groupby(
    ['region_id', 'region_name', 'station_id', 'station_name'],
    group_keys=True
).apply(calculate_eto_hargreaves)

eto_df_blaney_criddle = data.groupby(
    ['region_id', 'region_name', 'station_id', 'station_name'],
    group_keys=True
).apply(calculate_eto_blaney_criddle)

eto_df_oudin = data.groupby(
    ['region_id', 'region_name', 'station_id', 'station_name'],
    group_keys=True
).apply(calculate_eto_oudin)

eto_df_thornthwaite = data.groupby(
    ['region_id', 'region_name', 'station_id', 'station_name'],
    group_keys=True
).apply(calculate_eto_thornthwaite)

eto_df_makkink = data.groupby(
    ['region_id', 'region_name', 'station_id', 'station_name'],
    group_keys=True
).apply(calculate_eto_makkink)

# Join Monthly Mean ETo with Original Data

In [ ]:
if isinstance(eto_df_pm_asce.index, pd.MultiIndex):
    eto_df_pm_asce = eto_df_pm_asce.reset_index().rename(columns={0:'ASCE_PM'})
    data = pd.merge(data, eto_df_pm_asce, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
else:
    data = data.join(eto_df_pm_asce)

if isinstance(eto_df_pm_fao56.index, pd.MultiIndex):
    eto_df_pm_fao56 = eto_df_pm_fao56.reset_index().rename(columns={0:'FAO56'})
    data = pd.merge(data, eto_df_pm_fao56, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
else:
    data = data.join(eto_df_pm_fao56)

if isinstance(eto_df_hargreaves.index, pd.MultiIndex):
    eto_df_hargreaves = eto_df_hargreaves.reset_index().rename(columns={0:'Hargreaves'})
    data = pd.merge(data, eto_df_hargreaves, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
else:
    data = data.join(eto_df_hargreaves)

if isinstance(eto_df_blaney_criddle.index, pd.MultiIndex):
    eto_df_blaney_criddle = eto_df_blaney_criddle.reset_index().rename(columns={0:'Blaney_Criddle'})
    data = pd.merge(data, eto_df_blaney_criddle, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
else:
    data = data.join(eto_df_blaney_criddle)

if isinstance(eto_df_oudin.index, pd.MultiIndex):
    eto_df_oudin = eto_df_oudin.reset_index().rename(columns={0:'Oudin'})
    data = pd.merge(data, eto_df_oudin, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
else:
    data = data.join(eto_df_oudin)

if isinstance(eto_df_thornthwaite.index, pd.MultiIndex):
    eto_df_thornthwaite = eto_df_thornthwaite.reset_index().rename(columns={0:'Thornthwaite'})
    data = pd.merge(data, eto_df_thornthwaite, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
else:
    data = data.join(eto_df_oudin)
    
if isinstance(eto_df_makkink.index, pd.MultiIndex):
    eto_df_makkink = eto_df_makkink.reset_index().rename(columns={0:'Makkink'})
    data = pd.merge(data, eto_df_makkink, on=['region_id', 'region_name', 'station_id', 'station_name', 'date'], how='left')
else:
    data = data.join(eto_df_makkink)

data

# Convert Daily Mean ETo to Monthly Total ETo

In [ ]:
# Multiple number of day in month in FAO56, Hargreaves, Blaney_Criddle, Oudin
days_in_month = data['date'].dt.daysinmonth
data['ASCE_PM'] = data['ASCE_PM'] * days_in_month
data['FAO56'] = data['FAO56'] * days_in_month
data['Hargreaves'] = data['Hargreaves'] * days_in_month
data['Blaney_Criddle'] = data['Blaney_Criddle'] * days_in_month
data['Oudin'] = data['Oudin'] * days_in_month
data['Thornthwaite'] = data['Thornthwaite'] * days_in_month
data['Makkink'] = data['Makkink'] * days_in_month
data['date'] = pd.to_datetime(data[['year', 'month']].assign(day=1))
data = data.round(2)
data

# Plot ETo Time Series

In [ ]:
stations_to_plot = data['station_id'].drop_duplicates().sample(1)

plt.figure(figsize=(12, 6))
for sid in stations_to_plot:
    station_data = data[data['station_id'] == sid].sort_values('date')
    plt.plot(station_data['date'], station_data['FAO56'], label=f'Station {sid} - FAO56')
    plt.scatter(station_data['date'], station_data['FAO56'], color='blue', label=f'Station {sid} - FAO56', s=8)
    plt.scatter(station_data['date'], station_data['Hargreaves'], color='red', label=f'Station {sid} - Hargreaves', s=8)
    # plt.scatter(station_data['date'], station_data['Blaney_Criddle'], color='green', label=f'Station {sid} - Blaney_Criddle', s=8)
    # plt.scatter(station_data['date'], station_data['Oudin'], color='purple', label=f'Station {sid} - Oudin', s=8)
    plt.scatter(station_data['date'], station_data['Thornthwaite'], color='black', label=f'Station {sid} - Thornthwaite', s=8)
    plt.title(f'ETo Time Series for {station_data["region_name"].iloc[0]} - {station_data["station_name"].iloc[0]}')
    
plt.xlabel('Date')
plt.ylabel('ETo')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# # To pickle in data folder
# with open('../../data/Iran_Monthly_ETo_1951_2025.pkl', 'wb') as f:
#     pickle.dump(data, f)

# To parquet in data folder
data.to_parquet('../../data/Iran_Monthly_ETo_1951_2025.parquet')
logging.info("Monthly ETo data saved to pickle and parquet files.")